# Clean and preprocess Megas

In [117]:
import pandas as pd
import re
import string
from tqdm import tqdm


In [118]:
from bs4 import BeautifulSoup
import glob
import os

For Megas part of the preprocessing was simpler as we removed footnotes and marked the quotes manually beforehand

In [167]:
from pathlib import Path

#import manually marked text
folder = Path("Megas_quotes")

files = sorted(folder.glob("*.txt"))

files

[WindowsPath('Megas_quotes/Megas1_marked.txt'),
 WindowsPath('Megas_quotes/Megas2_marked.txt'),
 WindowsPath('Megas_quotes/Megas3_marked.txt')]

In [168]:
data = []

for file in files:
    text = file.read_text(encoding="utf-8")
    
    data.append({
        "file": file.stem,
        "Text": text
    })

df = pd.DataFrame(data)



In [169]:
#display the texts better
pd.set_option('display.width', 50)        
pd.set_option('display.max_colwidth', 50)

In [170]:
df.head()

,file,Text
0,Megas1_marked,﻿Προοίμιον. Όταν ο Όμηρος επεκαλείτο τήν Μούσα...
1,Megas2_marked,§ 19. Η φάλαγξ τών Κοραϊστών. Κατεπολέμησεν ο ...
2,Megas3_marked,"§ 34. Ιω Ν Ψυχάρη Ψευδοδημοτικισμός 1886. ""Θέλ..."


In [171]:
#df['text_clean'][0]

In [172]:
#display the texts better
pd.set_option('display.width', None)        
pd.set_option('display.max_colwidth', None)

### Remove newline characters

In [173]:
import unicodedata


In [174]:
def remove_newlines(text):
    """
    Removes '-\n' and replaces all other newlines with a space.
    """
    if not isinstance(text, str):
        return text

    # Remove hyphen-newlines (join split words)
    text = re.sub(r'-\n', '', text)

    # Replace all remaining newlines with a space
    text = text.replace('\n', ' ')

    # Normalize any multiple spaces to a single space
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [176]:
# Apply the remove_diacritics function to each entry in the 'Text' column
df['text_clean'] = df['Text'].apply(remove_newlines)

## Join pages per chapter

In [177]:
# Concatenate all sentences into a single text block
text_to_join = " ".join(df["text_clean"].astype(str))
#text_to_join

#### Split the text in chapters

In [178]:
chapter_markers = (
    ["Προοίμιον."]
    + [f"§ {i}." for i in range(1, 39)]
    + ["ΕΠΙΛΟΓΟΣ."]
)

In [179]:
chapter_markers

['Προοίμιον.',
 '§ 1.',
 '§ 2.',
 '§ 3.',
 '§ 4.',
 '§ 5.',
 '§ 6.',
 '§ 7.',
 '§ 8.',
 '§ 9.',
 '§ 10.',
 '§ 11.',
 '§ 12.',
 '§ 13.',
 '§ 14.',
 '§ 15.',
 '§ 16.',
 '§ 17.',
 '§ 18.',
 '§ 19.',
 '§ 20.',
 '§ 21.',
 '§ 22.',
 '§ 23.',
 '§ 24.',
 '§ 25.',
 '§ 26.',
 '§ 27.',
 '§ 28.',
 '§ 29.',
 '§ 30.',
 '§ 31.',
 '§ 32.',
 '§ 33.',
 '§ 34.',
 '§ 35.',
 '§ 36.',
 '§ 37.',
 '§ 38.',
 'ΕΠΙΛΟΓΟΣ.']

In [180]:
chapters = []

for i, marker in enumerate(chapter_markers):

    start = text_to_join.find(marker)

    if start == -1:
        raise ValueError(f"Δεν βρέθηκε το marker: {marker}")

    start = start + len(marker)

    # for all except Επίλογο
    if i < len(chapter_markers) - 1:
        next_marker = chapter_markers[i + 1]
        end = text_to_join.find(next_marker, start)

        if end == -1:
            raise ValueError(f"Δεν βρέθηκε το επόμενο marker: {next_marker}")

    # For Επίλογο
    else:
        final_phrase = "Ιδού η μέση οδός!"
        end = text_to_join.find(final_phrase, start)

        if end == -1:
            raise ValueError("Δεν βρέθηκε η τελευταία φράση.")

        # include final phrase 
        end += len(final_phrase)

    chapter_text = text_to_join[start:end].strip()

    chapters.append({
        "Chapters": marker,
        "Text": chapter_text
    })

In [181]:
# Create a DataFrame
df1 = pd.DataFrame(chapters)

In [182]:
#display the texts better
pd.set_option('display.width', 50)        
pd.set_option('display.max_colwidth', 500)

In [183]:
df1

,Chapters,Text
0,Προοίμιον.,"Όταν ο Όμηρος επεκαλείτο τήν Μούσαν, ένα τόν βοηθήση νά ψάλη τήν μήνεν τού Πηλείδου Αχιλλέως τήν ελεθρίαν, υπέβαλε πρός αυτήν δικαίαν καί ευλογον ερώτησιν τις τ άρ σφώε θεών έριδι ξυνέηκε μάχεσθαι; Ημείς όλως πεζοί καί πεζώς περί τού γλωσσικού ζητήματος γράφοντες, δέν έχομεν νά επικαλεσθώμεν τόν Θεόν ή άγγελον, ένα μάς φωτίση, εις όμηρικόν έπος, νά ψάλωμεν τήν μήνιν τού λογίου Έλληνος, ήτις προεκάλεσε τόν αιώνιον παρ ημίν περί γλώσσης γραφιδοπόλεμον καί μυρί άλγεα έφερε καί πρό πολλού καί χθ..."
1,§ 1.,"Ευγένιος ο Βούλγαρις υποκινητής τού γλωσσικού ζητήματος 1769 μ.Χ. Ευγένιος ο Βούλγαρις, ανήρ μαθύνους καί ευρυμαθής, καί εγκρατής τής Ελληνικής καί Λατινικής φιλολογίας, παρακολουθών τήν πρόοδον τής δημοτικής ημών γλώσσης. έχαιρε βεβαίως βλέπων εξαπλούμενον τό φώς τής παιδείας εις τάς κατωτέρας τάξεις τού λαού. Όταν όμως είδεν, ότι κατά τό παράδειγμα τού Σκούφου καί τού Δαμωδού, οίτινες εις δημώδη γλώσσαν έγραψαν τέχνην καί κανόνας ρητορικής, άλλοι εις τοιαύτην γλώσσαν ήρχι σαν νά γράφουν κα..."
2,§ 2.,"Ιώσηπος Μοισιόδαξ πρώτος απολογητής τής Νεοελληνικής γλώσσης 1761 1781. Μαθητής τού Ευγενίου, πολυμαθής ανήρ, ολιγώτερον όμως γνωστός εις τό πανελλήνιον, ο ιερομόναχος Ιώσηπος Μοισιόδας, σχολάρχης τής εν Ιασίω Ακαδημίας, προφαίνεται ήδη όχι μόνον συνήγορος, αλλά καί θερμός απολογητής τής δημοτικής γλώσσης εναντίον τής γνώμης τού σοφού διδασκάλου του, όστις διά τά φιλοσοφικά συγγράμματα εκήρυττεν αξίαν μόνην τήν αρχαίαν ελληνικήν. Τώ 1761 μεταφράσας ο Μοισιόδαξ τήν Ηθικήν Φιλοσοφίαν τού Ιταλο..."
3,§ 3.,"Δ. Καταρτζή καί Λάμπρου Φωτιάδου σπουδαία περί γλώσσης συζήτησις 1789. Πεδίον τού γλωσσικού αγώνος δέν είναι ακόμη επί πολύν χρόνον αι κλειναί Αθήναι η παχυλή ομίχλη, η οποία είχεν εξαπλωθή καθ όλον τόν Αττικόν ορίζοντα καί τήν παλαιάν Ελλάδα επί πολλούς αιώνας δέν άφηνε ν αναφανή τό θαλπερόν φώς εις μόρφωσιν τής γλώσσης καί παιδείας τού Γένους. Αι Ελικωνιάδες Μούσαι, φυγαδευθείσαι μετά τήν Αλωσιν εις τήν Ιταλίαν καί τήν λοιπήν Ευρώπην καί ήδη επανερχόμεναι εις τό πάτριον έδαφος, εζήτησαν ως..."
4,§ 4.,"Νικηφόρος Θεοτόκης ευδόκιμος αναμορφωτής υπερκαθαρευούσης γλώσσης 1796. Είναι αληθώς θείον δώρον η εμφάνισις καί συνεργασία πολλών καί σοφών ανδρών κατά τήν ηώ τής Εθνικής ημών παλεγγενεσίας. Εις τόν υπέρ τής γλώσσης αγώνα Ευγενίου τού Βουλγάρεως, τού Μοισιόδακος, τού Λάμπρου Φωτιάδου προφαίνεται καί άλλος όμοιος συναθλητής, ο εκ Κερκύρας Νικηφόρος Θεοτόκης, μέγας ανήρ, συνενών τήν πολυμάθειαν μέ βαθείαν γνώσιν τής αρχαίας καί τής νεωτέρας ημών γλώσσης . Όπως ο Μοισιόδαξ εις τά πρώτά του συγ..."
5,§ 5.,"Ρήγα τού Φεραίου ο Θούριος καί μετάφρασις δράματος εις απλήν καθαρεύουσαν 1797. 1. Προοίμια δέν χρειάζονται, ίνα γνωρίσωμεν εις τό κοινόν τών παρ ημίν λογίων τόν Πρωτομάρτυρα τής Εθνικής ημών Παλιγγενεσίας, τόν πρώτον βάρδον τόν εξεγείραντα τό φρόνημα τών δούλων αδελφών υπέρ τής ελευθερίας επί ένα όλον αιώνα μέ τόν ενθουσιώθη Θούριον Ως πότε παλληκάρια νά ζούμεν στά στενά είναι πανελλήνιον, αθάνατον τό όνομα Ρήγα. τού Φεραίου. Δέν είναι παράδοξον, άν εις τήν Ιστορίαν τού Γλωσσικού ζητήματος ..."
6,§ 6.,"Γρηγόριος Κωνσταντάς καί Δανιήλ Φιλιππίδης νέοι τού χυδαϊσμού υπέρμαχοι 1791 1818. Οι πολυετείς κόποι τού Καταρτζή δέν εκαρποφόρησαν τόσον, ώστε νά γίνη τό καινοτόμον γλωσσικόν του ιδίωμα Εθνική γλώσσα, αλλά καί δέν απέθανεν ούτος άνευ κληρονόμων τών παραδόξων αρχών του. Τό ύφος τής γλώσσης του μετ ολίγα έτη ασπάζονται δύο εκ Δημητριάδος τής Θεσσαλίας λόγιοι, ο ιεροδιάκονος Γρηγόριος Κωνσταντάς καί ο ιερομόναχος Δανιήλ Φιλιππίδης. Ούτοι τώ 1791 συγγράψαντες Νεωτερικήν Γεωγραφίαν εμφανίζονται..."
7,§ 7.,"Γρ Κωνσταντά μετριοπάθεια εις βελτίωσιν τής γλώσσης 1804 . Όπως τού Καταρτζή η καινονομία περί τήν γλώσσαν εξήγειρε τήν αποδοκιμασίαν τών τότε λογίων, ούτω καί τού Κωνσταντά καί Φιλιππίδου ο δημοτικισμός απήρεσεν εις τούς πολλούς καί εμυκτηρίσθη ως γλώσσα τής Ζαγοράς . Η τοιαύτη γενική καταδίκη κατέστησε τόν Κωνσταντάν προσεκτικώτερον δέν ενόμισεν ως προσβολήν ή έγκλημα, άν έδιδε προσοχήν εις τάς κρίσεις 

In [184]:
replacements = {
    "Προοίμιον.": "Preface",
    "§ 1.": "Chapter_1",
    "§ 2.": "Chapter_2",
    "§ 3.": "Chapter_3",
    "§ 4.": "Chapter_4",
    "§ 5.": "Chapter_5",
    "§ 6.": "Chapter_6",
    "§ 7.": "Chapter_7",
    "§ 8.": "Chapter_8",
    "§ 9.": "Chapter_9",
    "§ 10.": "Chapter_10",
    "§ 11.": "Chapter_11",
    "§ 12.": "Chapter_12",
    "§ 13.": "Chapter_13",
    "§ 14.": "Chapter_14",
    "§ 15.": "Chapter_15",
    "§ 16.": "Chapter_16",
    "§ 17.": "Chapter_17",
    "§ 18.": "Chapter_18",
    "§ 19.": "Chapter_19",
    "§ 20.": "Chapter_20",
    "§ 21.": "Chapter_21",
    "§ 22.": "Chapter_22",
    "§ 23.": "Chapter_23",
    "§ 24.": "Chapter_24",
    "§ 25.": "Chapter_25",
    "§ 26.": "Chapter_26",
    "§ 27.": "Chapter_27",
    "§ 28.": "Chapter_28",
    "§ 29.": "Chapter_29",
    "§ 30.": "Chapter_30",
    "§ 31.": "Chapter_31",
    "§ 32.": "Chapter_32",
    "§ 33.": "Chapter_33",
    "§ 34.": "Chapter_34",
    "§ 35.": "Chapter_35",
    "§ 36.": "Chapter_36",
    "§ 37.": "Chapter_37",
    "§ 38.": "Chapter_38",
    "ΕΠΙΛΟΓΟΣ.": "Epilogue"
}
   

In [185]:
# replace words based on the designated dictionary
def replace_words(text, replacements):
    for incorrect, correct in replacements.items():
        text = text.replace(incorrect, correct)
    return text


In [186]:
df1['Chapters'] = df1['Chapters'].apply(lambda x: replace_words(str(x), replacements))

In [187]:
df1

,Chapters,Text
0,Preface,"Όταν ο Όμηρος επεκαλείτο τήν Μούσαν, ένα τόν βοηθήση νά ψάλη τήν μήνεν τού Πηλείδου Αχιλλέως τήν ελεθρίαν, υπέβαλε πρός αυτήν δικαίαν καί ευλογον ερώτησιν τις τ άρ σφώε θεών έριδι ξυνέηκε μάχεσθαι; Ημείς όλως πεζοί καί πεζώς περί τού γλωσσικού ζητήματος γράφοντες, δέν έχομεν νά επικαλεσθώμεν τόν Θεόν ή άγγελον, ένα μάς φωτίση, εις όμηρικόν έπος, νά ψάλωμεν τήν μήνιν τού λογίου Έλληνος, ήτις προεκάλεσε τόν αιώνιον παρ ημίν περί γλώσσης γραφιδοπόλεμον καί μυρί άλγεα έφερε καί πρό πολλού καί χθ..."
1,Chapter_1,"Ευγένιος ο Βούλγαρις υποκινητής τού γλωσσικού ζητήματος 1769 μ.Χ. Ευγένιος ο Βούλγαρις, ανήρ μαθύνους καί ευρυμαθής, καί εγκρατής τής Ελληνικής καί Λατινικής φιλολογίας, παρακολουθών τήν πρόοδον τής δημοτικής ημών γλώσσης. έχαιρε βεβαίως βλέπων εξαπλούμενον τό φώς τής παιδείας εις τάς κατωτέρας τάξεις τού λαού. Όταν όμως είδεν, ότι κατά τό παράδειγμα τού Σκούφου καί τού Δαμωδού, οίτινες εις δημώδη γλώσσαν έγραψαν τέχνην καί κανόνας ρητορικής, άλλοι εις τοιαύτην γλώσσαν ήρχι σαν νά γράφουν κα..."
2,Chapter_2,"Ιώσηπος Μοισιόδαξ πρώτος απολογητής τής Νεοελληνικής γλώσσης 1761 1781. Μαθητής τού Ευγενίου, πολυμαθής ανήρ, ολιγώτερον όμως γνωστός εις τό πανελλήνιον, ο ιερομόναχος Ιώσηπος Μοισιόδας, σχολάρχης τής εν Ιασίω Ακαδημίας, προφαίνεται ήδη όχι μόνον συνήγορος, αλλά καί θερμός απολογητής τής δημοτικής γλώσσης εναντίον τής γνώμης τού σοφού διδασκάλου του, όστις διά τά φιλοσοφικά συγγράμματα εκήρυττεν αξίαν μόνην τήν αρχαίαν ελληνικήν. Τώ 1761 μεταφράσας ο Μοισιόδαξ τήν Ηθικήν Φιλοσοφίαν τού Ιταλο..."
3,Chapter_3,"Δ. Καταρτζή καί Λάμπρου Φωτιάδου σπουδαία περί γλώσσης συζήτησις 1789. Πεδίον τού γλωσσικού αγώνος δέν είναι ακόμη επί πολύν χρόνον αι κλειναί Αθήναι η παχυλή ομίχλη, η οποία είχεν εξαπλωθή καθ όλον τόν Αττικόν ορίζοντα καί τήν παλαιάν Ελλάδα επί πολλούς αιώνας δέν άφηνε ν αναφανή τό θαλπερόν φώς εις μόρφωσιν τής γλώσσης καί παιδείας τού Γένους. Αι Ελικωνιάδες Μούσαι, φυγαδευθείσαι μετά τήν Αλωσιν εις τήν Ιταλίαν καί τήν λοιπήν Ευρώπην καί ήδη επανερχόμεναι εις τό πάτριον έδαφος, εζήτησαν ως..."
4,Chapter_4,"Νικηφόρος Θεοτόκης ευδόκιμος αναμορφωτής υπερκαθαρευούσης γλώσσης 1796. Είναι αληθώς θείον δώρον η εμφάνισις καί συνεργασία πολλών καί σοφών ανδρών κατά τήν ηώ τής Εθνικής ημών παλεγγενεσίας. Εις τόν υπέρ τής γλώσσης αγώνα Ευγενίου τού Βουλγάρεως, τού Μοισιόδακος, τού Λάμπρου Φωτιάδου προφαίνεται καί άλλος όμοιος συναθλητής, ο εκ Κερκύρας Νικηφόρος Θεοτόκης, μέγας ανήρ, συνενών τήν πολυμάθειαν μέ βαθείαν γνώσιν τής αρχαίας καί τής νεωτέρας ημών γλώσσης . Όπως ο Μοισιόδαξ εις τά πρώτά του συγ..."
5,Chapter_5,"Ρήγα τού Φεραίου ο Θούριος καί μετάφρασις δράματος εις απλήν καθαρεύουσαν 1797. 1. Προοίμια δέν χρειάζονται, ίνα γνωρίσωμεν εις τό κοινόν τών παρ ημίν λογίων τόν Πρωτομάρτυρα τής Εθνικής ημών Παλιγγενεσίας, τόν πρώτον βάρδον τόν εξεγείραντα τό φρόνημα τών δούλων αδελφών υπέρ τής ελευθερίας επί ένα όλον αιώνα μέ τόν ενθουσιώθη Θούριον Ως πότε παλληκάρια νά ζούμεν στά στενά είναι πανελλήνιον, αθάνατον τό όνομα Ρήγα. τού Φεραίου. Δέν είναι παράδοξον, άν εις τήν Ιστορίαν τού Γλωσσικού ζητήματος ..."
6,Chapter_6,"Γρηγόριος Κωνσταντάς καί Δανιήλ Φιλιππίδης νέοι τού χυδαϊσμού υπέρμαχοι 1791 1818. Οι πολυετείς κόποι τού Καταρτζή δέν εκαρποφόρησαν τόσον, ώστε νά γίνη τό καινοτόμον γλωσσικόν του ιδίωμα Εθνική γλώσσα, αλλά καί δέν απέθανεν ούτος άνευ κληρονόμων τών παραδόξων αρχών του. Τό ύφος τής γλώσσης του μετ ολίγα έτη ασπάζονται δύο εκ Δημητριάδος τής Θεσσαλίας λόγιοι, ο ιεροδιάκονος Γρηγόριος Κωνσταντάς καί ο ιερομόναχος Δανιήλ Φιλιππίδης. Ούτοι τώ 1791 συγγράψαντες Νεωτερικήν Γεωγραφίαν εμφανίζονται..."
7,Chapter_7,"Γρ Κωνσταντά μετριοπάθεια εις βελτίωσιν τής γλώσσης 1804 . Όπως τού Καταρτζή η καινονομία περί τήν γλώσσαν εξήγειρε τήν αποδοκιμασίαν τών τότε λογίων, ούτω καί τού Κωνσταντά καί Φιλιππίδου ο δημοτικισμός απήρεσεν εις τούς πολλούς καί εμυκτηρίσθη ως γλώσσα τής Ζαγοράς . Η τοιαύτη γενική καταδίκη κατέστησε τόν Κωνσταντάν προσεκτικώτερον δέν ενόμισεν ως προσβολήν ή έγκλημα, άν

In [188]:
# quotes were marked with ?--> detect them
df1["question_mark_count"] = df1["Text"].apply(
    lambda x: x.count("?") if isinstance(x, str) else 0
)

df1["question_mark_pairs_ok"] = (
    df1["question_mark_count"] % 2 == 0
)

df1[
    ["Chapters", "question_mark_count", "question_mark_pairs_ok"]
]

,Chapters,question_mark_count,question_mark_pairs_ok
0,Preface,0,True
1,Chapter_1,8,True
2,Chapter_2,26,True
3,Chapter_3,14,True
4,Chapter_4,10,True
5,Chapter_5,10,True
6,Chapter_6,2,True
7,Chapter_7,10,True
8,Chapter_8,16,True
9,Chapter_9,26,True


In [189]:
df1.loc[
    ~df1["question_mark_pairs_ok"],
    ["Chapters", "question_mark_count"]
]

,Chapters,question_mark_count


In [190]:
# extract marked quotes to a new column

def extract_marked_quotes(text):
    if not isinstance(text, str):
        return ""

    quotes = re.findall(
        r"\?.*?\?",
        text,
        flags=re.DOTALL
    )

    return " ".join(q.strip() for q in quotes)


def remove_marked_quotes(text):
    if not isinstance(text, str):
        return ""

    text_no_quotes = re.sub(
        r"\?.*?\?",
        " ",
        text,
        flags=re.DOTALL
    )

    # clean extra spaces 
    text_no_quotes = re.sub(r"\s+", " ", text_no_quotes)

    return text_no_quotes.strip()

In [191]:
df1["quoted_text"] = df1["Text"].apply(extract_marked_quotes)

df1["text_no_quotes"] = df1["Text"].apply(remove_marked_quotes)

In [193]:
df1.head(2)

,Chapters,Text,question_mark_count,question_mark_pairs_ok,quoted_text,text_no_quotes
0,Preface,"Όταν ο Όμηρος επεκαλείτο τήν Μούσαν, ένα τόν βοηθήση νά ψάλη τήν μήνεν τού Πηλείδου Αχιλλέως τήν ελεθρίαν, υπέβαλε πρός αυτήν δικαίαν καί ευλογον ερώτησιν τις τ άρ σφώε θεών έριδι ξυνέηκε μάχεσθαι; Ημείς όλως πεζοί καί πεζώς περί τού γλωσσικού ζητήματος γράφοντες, δέν έχομεν νά επικαλεσθώμεν τόν Θεόν ή άγγελον, ένα μάς φωτίση, εις όμηρικόν έπος, νά ψάλωμεν τήν μήνιν τού λογίου Έλληνος, ήτις προεκάλεσε τόν αιώνιον παρ ημίν περί γλώσσης γραφιδοπόλεμον καί μυρί άλγεα έφερε καί πρό πολλού καί χθ...",0,True,,"Όταν ο Όμηρος επεκαλείτο τήν Μούσαν, ένα τόν βοηθήση νά ψάλη τήν μήνεν τού Πηλείδου Αχιλλέως τήν ελεθρίαν, υπέβαλε πρός αυτήν δικαίαν καί ευλογον ερώτησιν τις τ άρ σφώε θεών έριδι ξυνέηκε μάχεσθαι; Ημείς όλως πεζοί καί πεζώς περί τού γλωσσικού ζητήματος γράφοντες, δέν έχομεν νά επικαλεσθώμεν τόν Θεόν ή άγγελον, ένα μάς φωτίση, εις όμηρικόν έπος, νά ψάλωμεν τήν μήνιν τού λογίου Έλληνος, ήτις προεκάλεσε τόν αιώνιον παρ ημίν περί γλώσσης γραφιδοπόλεμον καί μυρί άλγεα έφερε καί πρό πολλού καί χθ..."
1,Chapter_1,"Ευγένιος ο Βούλγαρις υποκινητής τού γλωσσικού ζητήματος 1769 μ.Χ. Ευγένιος ο Βούλγαρις, ανήρ μαθύνους καί ευρυμαθής, καί εγκρατής τής Ελληνικής καί Λατινικής φιλολογίας, παρακολουθών τήν πρόοδον τής δημοτικής ημών γλώσσης. έχαιρε βεβαίως βλέπων εξαπλούμενον τό φώς τής παιδείας εις τάς κατωτέρας τάξεις τού λαού. Όταν όμως είδεν, ότι κατά τό παράδειγμα τού Σκούφου καί τού Δαμωδού, οίτινες εις δημώδη γλώσσαν έγραψαν τέχνην καί κανόνας ρητορικής, άλλοι εις τοιαύτην γλώσσαν ήρχι σαν νά γράφουν κα...",8,True,"?Εκσυρικτέον? ?τά χυδαϊστί φιλοσοφείν επαγγελλόμενα βιβλιδάρια, τής Ελλάδος φωνής ως οίόν τε επιμελομένους, ής άνευ ουδέν τών πάλαι πεφιλοσοφηκότων εστίν απόνασθαι.? ?Πολλούς καί ποικίλους τρόπους μεταχειρίζονται αυτοί οι Απόστολοι τής Ρώμης διά νά αρπάσουν από τάς αγκάλας τής Ορθοδόξου Ανατολικής Εκκλησίας εις τόν παπισμόν τά γνήσια τέκνα της είναι δυνατόν όμως τούς πολλούς τρόπους νά τούς ανακεφαλαιώση τις εις τρείς, διά νά προσέχη κάθε ένας καί νά φυλάττηται. Έρχονται αυτοί εις τόν αγώνα ...","Ευγένιος ο Βούλγαρις υποκινητής τού γλωσσικού ζητήματος 1769 μ.Χ. Ευγένιος ο Βούλγαρις, ανήρ μαθύνους καί ευρυμαθής, καί εγκρατής τής Ελληνικής καί Λατινικής φιλολογίας, παρακολουθών τήν πρόοδον τής δημοτικής ημών γλώσσης. έχαιρε βεβαίως βλέπων εξαπλούμενον τό φώς τής παιδείας εις τάς κατωτέρας τάξεις τού λαού. Όταν όμως είδεν, ότι κατά τό παράδειγμα τού Σκούφου καί τού Δαμωδού, οίτινες εις δημώδη γλώσσαν έγραψαν τέχνην καί κανόνας ρητορικής, άλλοι εις τοιαύτην γλώσσαν ήρχι σαν νά γράφουν κα..."


In [194]:
df1.drop(
    columns=["question_mark_pairs_ok", "question_mark_count"],
    inplace=True
)

In [195]:
df1 = df1[["Chapters", "Text", "text_no_quotes", "quoted_text"]]

In [196]:
#display the sentences better
pd.set_option('display.width', None)        
pd.set_option('display.max_colwidth', None)

In [ ]:
#check
filtered_text = df1[df1['Chapters'] == 'Chapter_38']
filtered_text

In [198]:
# basic cleaning

def basic_clean(text):
    if not isinstance(text, str):
        return ""

    # Πολλαπλά whitespace -> ένα κενό
    text = re.sub(r"\s+", " ", text)

    # Αφαίρεση κενού πριν από σημεία στίξης
    text = re.sub(r"\s+([,.;:!])", r"\1", text)

    return text.strip()

In [199]:
df1["Text"] = df1["Text"].apply(basic_clean)

In [200]:
df1["text_no_quotes"] = df1["text_no_quotes"].apply(basic_clean)

### Lets count words

In [201]:

def count_words(text):
    if not isinstance(text, str):
        return 0

    words = re.findall(
        r"[^\W\d_]+(?:['’᾿᾽][^\W\d_]*)?",
        text,
        flags=re.UNICODE
    )

    return len(words)

In [203]:
df1["Original_Text_words"] = df1["Text"].apply(count_words)
df1["text_no_quote_words"] = df1["text_no_quotes"].apply(count_words)
df1["quoted_text_words"] = df1["quoted_text"].apply(count_words)

In [204]:
df1["check"] = (
    df1["text_no_quote_words"]
    + df1["quoted_text_words"]
    - df1["Original_Text_words"]
)

df1["check"].value_counts()

check
0    40
Name: count, dtype: int64

In [205]:
total_words = df1["Original_Text_words"].sum()
total_text_no_quote_words = df1["text_no_quote_words"].sum()
total_quoted_text_words = df1["quoted_text_words"].sum()

In [206]:
print(f"TOTAL WORD COUNT: {total_words:,}")
print(f"NO QUOTES TEXT WORD COUNT: {total_text_no_quote_words:,}")
print(f"QUOTED TEXT WORD COUNT: {total_quoted_text_words:,}")
print("Own %:", total_text_no_quote_words / total_words * 100)
print("Quoted %:", total_quoted_text_words / total_words * 100)

TOTAL WORD COUNT: 176,175
NO QUOTES TEXT WORD COUNT: 49,777
QUOTED TEXT WORD COUNT: 126,398
Own %: 28.254292606783025
Quoted %: 71.74570739321697


In [207]:
df1.to_csv("Megas_clean_counted.csv", index=False)

### Reimport data

In [2]:
#df1 = pd.read_csv('Megas_clean_counted.csv')


### Convert to monotonic

In [209]:
import unicodedata

def convert_polytonic_to_simple(text):
    if not isinstance(text, str):
        return ""
    """
    Removes (δασεία and ψιλή) and converts περισπωμένη to oξεία.
    
    Args:
        text (str): Polytonic Greek text.
        
    Returns:
        str: Simplified text with πνευματα removed and περισπωμένη converted to ὀξεία.
    """
    # Normalize the text to NFD (decomposed form)
    text = unicodedata.normalize('NFD', text)
    
    # Remove breathing marks: δασεία (◌̔) and ψιλή (◌̓)
    text = re.sub(r'[\u0313\u0314]', '', text)  # Removes U+0313 (ψιλή) and U+0314 (δασεία)
    
    # Replace περισπωμένη (◌̃) with ὀξεία (◌́)
    text = re.sub(r'\u0342', '\u0301', text)  # Replace U+0342 (περισπωμένη) with U+0301 (ὀξεία)
    
    # Normalize back to NFC (composed form)
    text = unicodedata.normalize('NFC', text)
    
    return text


In [210]:
# Example 
polytonic_text = """«Τὸ ἅπλωμα τοῦ δημοτικισμοῦ Δημοτικὴ καὶ φιλολογία Ὕστερ ἀπὸ τὰ Βῤαγγελιχὰ καὶ τὰ Ὀρεστειακά ἡ γλωσσικὴ ἀντίδραση προσπάθησε μὲ ὅπλο τὴ συχοφαντία νἀ ἐμποδίσει τὸ ἄπλωμα τοῦ δημοτικισμοῦ.'"""
simplified_text = convert_polytonic_to_simple(polytonic_text)

print("Original Text:")
print(polytonic_text)
print("\nSimplified Text:")
print(simplified_text)

Original Text:
«Τὸ ἅπλωμα τοῦ δημοτικισμοῦ Δημοτικὴ καὶ φιλολογία Ὕστερ ἀπὸ τὰ Βῤαγγελιχὰ καὶ τὰ Ὀρεστειακά ἡ γλωσσικὴ ἀντίδραση προσπάθησε μὲ ὅπλο τὴ συχοφαντία νἀ ἐμποδίσει τὸ ἄπλωμα τοῦ δημοτικισμοῦ.'

Simplified Text:
«Τὸ άπλωμα τού δημοτικισμού Δημοτικὴ καὶ φιλολογία Ύστερ απὸ τὰ Βραγγελιχὰ καὶ τὰ Ορεστειακά η γλωσσικὴ αντίδραση προσπάθησε μὲ όπλο τὴ συχοφαντία να εμποδίσει τὸ άπλωμα τού δημοτικισμού.'


In [212]:
# Apply the function to simplify polytonic Greek
df1['text_monotonic'] = df1['Text'].apply(convert_polytonic_to_simple)

In [213]:
df1['quoted_text_monotonic'] = df1['quoted_text'].apply(convert_polytonic_to_simple)

In [214]:
df1['text_no_quotes_monotonic'] = df1['text_no_quotes'].apply(convert_polytonic_to_simple)

In [1]:
# check
df1[:2]

### Fix word spelling with frequency analysis

In [217]:
from nltk.tokenize import word_tokenize
import itertools
from collections import Counter


In [ ]:
def tokenize(text):
    # Tokenize without lowercasing
    tokens = word_tokenize(text, language='greek')  # Specify Greek for correct tokenization
    #tokens = word_tokenize(text.lower(), language='greek')  # Specify Greek for correct tokenization
    return tokens


In [221]:
# Apply tokenization to our dataset
df1['tokens'] = df1['text_no_quotes_monotonic'].apply(tokenize)

In [222]:
#display the texts better
pd.set_option('display.width', 150)        
pd.set_option('display.max_colwidth', 150)

In [224]:
# Efficiently concatenate all token lists into a single list
all_tokens = list(itertools.chain.from_iterable(df1['tokens']))

# Count word frequencies
word_counts = Counter(all_tokens)

# Example to print the frequency of the word 'text'
#print(word_counts['απλοελληνικη'])  # Replace 'text' with 'απλη' or any Greek word you're interested in
#print(word_counts['διαλεκτον']) 
print(word_counts['και']) 


14


In [225]:
# check the most common words
print(word_counts.most_common(800))

[(',', 3599), ('καί', 2621), ('τού', 1429), ('εις', 1234), ('τήν', 1173), ('.', 1116), ('τής', 944), ('ο', 831), ('τό', 793), ('τών', 708), ('τόν', 681), ('νά', 558), ('τά', 479), ('δέν', 434), ('η', 411), ('ότι', 391), ('πρός', 360), ('ως', 340), ('γλώσσης', 308), ('του', 308), ('είναι', 299), ('δέ', 272), ('γλώσσαν', 256), ('τούς', 233), ('οι', 225), ('τάς', 202), ('διά', 187), ('μας', 184), ('από', 183), ('μέ', 179), ('θά', 170), ('περί', 165), ('ή', 162), ('άν', 158), (';', 154), ('αυτού', 150), ('κατά', 145), ('όμως', 141), ('εν', 140), ('εκ', 121), ('ημών', 120), ('!', 112), ('γλώσσα', 106), ('αλλά', 101), ('όπως', 100), ('Καί', 99), ('σήμερον', 98), ('τούτο', 98), ('παρ', 89), ('ν', 87), ('όχι', 86), ('Κοραή', 84), ('μόνον', 83), ('αλλ', 82), ('διότι', 81), ('των', 78), ('αυτός', 75), ('Δέν', 73), ('οποίον', 73), ('Ο', 71), (':', 71), ('λογίων', 70), ('μή', 70), ('τώ', 68), ('ημίν', 66), ('αυτήν', 64), ('μάς', 64), ('αυτής', 64), ('υπό', 62), ('ήτο', 61), ('αι', 60), ('οποίαν', 

In [40]:
#  concatenate all token lists into a single list
all_tokens = list(itertools.chain.from_iterable(df1['tokens']))

# Count word frequencies
word_counts = Counter(all_tokens)

# Extract words with frequency less than 8
least_common_words = {word: freq for word, freq in word_counts.items() if freq < 7}

# Sort least common words from more frequent to less frequent
sorted_least_common = sorted(least_common_words.items(), key=lambda x: x[1], reverse=True)


In [ ]:
# Print sorted least common words
print("Words with frequency less than 7 (sorted):")
for word, freq in sorted_least_common:
    print(f"{word}: {freq}")

### Check 'broken' words

In [226]:
def get_partial_context(text, target, window=5):
    """
    Finds the context around a target substring in the text, 
    including cases where words are split.

    Parameters:
    text (str): The text to search.
    target (str): The specific substring to find (partial match).
    window (int): The number of words before and after the target.

    Returns:
    list: A list of contexts where the target substring appears.
    """
    # Regex to match the target substring and surrounding words
    pattern = rf'((?:\S+\s+){{0,{window}}})(\S*{re.escape(target)}\S*)(?:\s+(\S+)){{0,{window}}}'
    
    # Find all matches
    matches = re.finditer(pattern, text)
    
    # Collect the contexts
    contexts = []
    for match in matches:
        before = match.group(1).strip()  # Words before the target
        target_word = match.group(2).strip()  # Word containing the target
        after = match.group(3) or ''  # Words after the target (if any)
        contexts.append(f"{before} {target_word} {after}".strip())
    
    return contexts

# Example 
text = "Έξω απ αυτή χτίζουμε καί θά χτί ζουμε στήν άμμο ."

# Specify the target substring
target = "χτί "

# Get the contexts for the target
contexts = get_partial_context(text, target, window=10)

# Print the results
print("Contexts found:")
for context in contexts:
    print(context)

Contexts found:
Έξω απ αυτή χτίζουμε καί θά χτί ζουμε .


In [228]:
# Specify the target string
target = " και "

# Collect all contexts across the DataFrame
all_contexts = []
for text in df1['text_no_quotes_monotonic']:
    contexts = get_partial_context(text, target, window=10)
    all_contexts.extend(contexts)

# Print the results in a clean list format
print("Contexts found:")
for context in all_contexts:
    print(context)

Contexts found:
τών λογίων τού Έθνους, εν ώ συγχρόνως εις πόλεις καί όρη και βουνά παλληκάρια
παραδόσεως ορθο γραφίαν τής γραφομένης γλώσσης καί νά γράφωμεν απλά καί και Μαρά τον
μέ επιγρατήν Η ρομεηκη γλοσα ή Μηκρη ορμηνια για τα γραμματα και τη Χριστοπούλου
ομοιοκαταληξίας η φύσις ν αφίσης. 3 Άν δέ ο Σολωμός μετεχειρίσθη και τινας όπως
επέτυχε νά επιβληήν η καλλικέλαδος Μούσα μεγάλων Κρητών ποιητών, η απλή͵ και αφελής, Μηνιάτου,
παραδόξων αρχαϊστών Καί όμως μάς έμεινε καί εξ αυτού κάτι μέγα και θαυμαστόν εις
υβρίζων, ηξυνήθη νά υφαρπάση͵ ως οικειότερον καί τιμιώτερον δι εκυτόν Ελευθερία και Γλώσσα.
πρόβατα πού πήραμε άσχημο δρόμο ως πρός τήν γλώσσαν καί παιδείαν και λέγει κατά
τήν πλήρη εθνικήν ένωσιν. Αυτά λοιπόν τά πράγματα φωνάζουν έρχου, φίλε, και ιδέ εις
ο σοφός καθηγητής Χατζηδάκις καί τήν άλλην ομολογίαν τού Κονεμένου ότι και παρατηρεί διατριβήν
άσμα, ο αρχιμουσικός έλεγε Τό τραγούδι Βέη μου τό λέγει. Διότι και βέηδες τή
τού Τοξότου Θεοδώρου Ορφανίδου (1815-1886): 3 Εκ τών ποι

In [229]:
replacements = {
    r'\bηρ νούντο\b': 'ηρνούντο',
    r'\bαρι χαΐζον\b': 'αρχαΐζον',
    r'\bαρι χαϊζόντων\b': 'αρχαϊζόντων',
    r'\bαρι χαϊσμού\b': 'αρχαϊσμού',
    r'\bαρι χαίας\b': 'αρχαίας',
    r'\bαρι στερόν\b': 'αριστερόν',
    r'\bαρχαί κώτερον\b': 'αρχαϊκώτερον',
    r'\bγρά φει\b': 'γράφει',
    r'\bγρά φωμεν\b': 'γράφωμεν',
    r'\bγρά φουν\b': 'γράφουν',
    r'\bΑρά γε\b': 'Αράγε',
    r'\bγε νεών\b': 'γενεών',
    r'\bάρά γε\b': 'άραγε',
    r'\bσή μερον\b': 'σήμερον',
    r'\bσή μερα\b': 'σήμερα',
    r'\bζή τημα\b': 'ζήτημα',
    r'\bαντιπά λου\b': 'αντιπάλου',
    r'\bλου γιωτατισμόν\b': 'λογιωτατισμόν',
    r'\bλου γικήν\b': 'λογικήν',
    r'\bγλωσ σικού\b': 'γλωσσικού',
    r'\bκαθα ρεύοντας\b': 'καθαρεύοντας',
     r'\bκαθαρευ ούσης\b': 'καθαρευούσης',
    "λογοτε χνίας": "λογοτεχνίας",
"δημοσιογρα φίαν": "δημοσιογραφίαν",
"χρό νων": "χρόνων",
"πο λιτικά": "πολιτικά",
"πό λεις":	"πόλεις",
"δι δάσκαλοι":	"διδάσκαλοι",
"γλωσσο μεταρρυθμιστών": "γλωσσομεταρρυθμιστών",
"γλώσ σαν":	"γλώσσαν",
"εκλέ γει":	"εκλέγει",
"φαν τασίαν": "φαντασίαν",
"λου γικήν": "λογικήν",
"στρώ ματα": "στρώματα",
"δευτε ρόλεπτα": "δευτερόλεπτα",
"πε ριεφρόνει": "περιεφρόνει",
"τοποθε τείται": "τοποθετείται",
"ψευ δολογιών": "ψευδολογιών",
"μαλ λιαρισμού": "μαλλιαρισμού",
"συντελε σταί": "συντελεσταί",
"προγονό πληχτους": "προγονόπληχτους", 

   
}

In [230]:
def replace_words2(text, replacements):
    for incorrect, correct in replacements.items():
        text = re.sub(incorrect, correct, text)
    return text

In [231]:
# Apply the replacements to the columns
df1['text_no_quotes_monotonic'] = df1['text_no_quotes_monotonic'].apply(lambda x: replace_words2(str(x), replacements))

In [232]:
df1['text_monotonic'] = df1['text_monotonic'].apply(lambda x: replace_words2(str(x), replacements))

### Remove accents from stopwords

In [237]:
pre_correction_dict = {
    'καί': 'και',
    'τού': 'του',
    'τό': 'το',
    'τής': 'της',
    'νά': 'να',
    'τήν': 'την',
    'τά': 'τα',
    'πού': 'που',
    'τούς': 'τους',
    'μέ': 'με',
    'δέν': 'δεν',
    'τών': 'των',
    'τή': 'τη',
    'γιά': 'για',
    'τόν': 'τον',
    'πώς': 'πως',
    'καί': 'και',
    'στό': 'στο',
    'στήν': 'στην',
    'τίς': 'τις',
    'στά': 'στα',
    'στή': 'στη',
    'μιά': 'μια',
    'Ό': 'Ο',
    'σέ': 'σε',
    'Ή': 'Η',
    'θά': 'θα',
    'δέ': 'δε',
    'μά': 'μα',
    'Τό': 'Το',
    'άν': 'αν',
    'Μά': 'Μα',
    'πιό': 'πιο',
    'Καί': 'Και',
    'τάς': 'τας',
    'στίς': 'στις',
    'στόν': 'στον',
    'στούς': 'στους',
    'πρός': 'προς',
    'σάν': 'σαν',
    'δυό': 'δυο',
    'Δέν': 'Δεν',
    'μή': 'μη',
    'Νά': 'Να',
    'μαζύ': 'μαζί',
    'Στήν': 'Στην',
    'σά': 'σα',
    'μήν': 'μην',
    'Σέ': 'Σε',
    'Στά': 'Στα',
    'πρίν': 'πριν',
    'τῴ': 'τω',
    'άς': 'ας',
    'μέν': 'μεν',
    'Τήν': 'Την',
    'Μέ': 'Με',
    'Άν': 'Αν',
    'τῄ': 'τη',
    'ό': 'ο',
    'είς': 'εις',
    'πρό': 'προ',
    'Μιά': 'Μια',
    'μιάς': 'μιας',
    'Γιά': 'Για',
    'ού': 'ου',
    'τοίς': 'τοις',
    'Στή': 'Στη',
    'τές': 'τες',
    'ή': 'η'
}

In [238]:
def pre_correct_stopwords_safe_case_sensitive(text, correction_dict):
    """
    Replaces stopwords with de-accented versions, preserving case.
    """
    # Compile a pattern for word boundaries and dictionary keys
    pattern = re.compile(r'\b(' + '|'.join(map(re.escape, correction_dict.keys())) + r')\b')
    
    def replace_with_case(match):
        """
        Replaces the matched word with the corresponding de-accented version,
        preserving the original case.
        """
        original_word = match.group(0)
        replacement_word = correction_dict[original_word]  # Get the de-accented version

        # Check if the original word was all uppercase or title case
        if original_word.isupper():
            return replacement_word.upper()  # Replace with uppercase
        elif original_word.istitle():
            return replacement_word.title()  # Replace with title case
        else:
            return replacement_word  # Replace with lowercase

    # Replace using the custom replacement function
    return pattern.sub(replace_with_case, text)


In [241]:
# Apply to the whole DataFrame
df1['text_no_quotes_monotonic'] = df1['text_no_quotes_monotonic'].apply(lambda x: pre_correct_stopwords_safe_case_sensitive(x, pre_correction_dict))

In [242]:
df1['text_monotonic'] = df1['text_monotonic'].apply(lambda x: pre_correct_stopwords_safe_case_sensitive(x, pre_correction_dict))

In [2]:
# Check
df1['text_monotonic'][3]

In [246]:
df1 = df1.drop(columns=['tokens'])

In [247]:
# beter be safe than sorry
df1.to_csv('Megas_analysis_ready.csv', index=False, encoding='utf-8-sig')